# Module Clustering
Power-quality and time-pattern based K-means clustering cleaned from the original local notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

df = pd.read_csv('factory_hourly_final.csv')
df['localtime'] = pd.to_datetime(df['localtime'])


In [ ]:
features_to_use = [
    'activePower','total_pf','tw_avg_pf_R','tw_avg_pf_S','tw_avg_pf_T',
    'tw_avg_reactivePow','current_CV','current_unbal_ratio',
    'current_change_acc','voltage_CV_RS','voltage_CV_ST','voltage_CV_TR'
]

module_stats = df.groupby('module(equipment)')[features_to_use].mean().fillna(0)
scaled_data = StandardScaler().fit_transform(module_stats)

sse = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init='k-means++', random_state=42)
    km.fit(scaled_data)
    sse.append(km.inertia_)

plt.plot(range(1, 11), sse, marker='o')
plt.xlabel('K'); plt.ylabel('SSE'); plt.title('Elbow Method')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42)
module_stats['cluster'] = kmeans.fit_predict(scaled_data)

pca = PCA(n_components=2)
xy = pca.fit_transform(scaled_data)
module_stats['pca_x'] = xy[:,0]
module_stats['pca_y'] = xy[:,1]
module_stats[['cluster','pca_x','pca_y']]


In [ ]:
def normalized_pattern(data, value_col):
    pivot = data.pivot_table(index='module(equipment)', columns='hour', values=value_col, aggfunc='mean').fillna(0)
    scaler = MinMaxScaler()
    arr = scaler.fit_transform(pivot.T).T
    return pd.DataFrame(arr, index=pivot.index, columns=[f'{value_col}_h{i}' for i in range(24)])

active_p = normalized_pattern(df, 'activePower')
reactive_p = normalized_pattern(df, 'tw_avg_reactivePow')
hourly_p = normalized_pattern(df, 'hourly_pow')

quality_features = ['total_pf','current_CV','current_unbal_ratio','current_change_acc']
module_quality = df.groupby('module(equipment)')[quality_features].mean().fillna(0)
combined = pd.concat([module_quality, active_p, reactive_p, hourly_p], axis=1)
combined_scaled = StandardScaler().fit_transform(combined)
combined['cluster'] = KMeans(n_clusters=4, random_state=42).fit_predict(combined_scaled)
combined[['cluster']]


## Final cluster interpretation
- Cluster 0: 2호기, 우측분전반2 — base-load / high reactive-power pattern
- Cluster 1: 3호기, 예비건조기, 6호기, 분쇄기1, 좌측분전반 — high-variance work-synchronous pattern
- Cluster 2: PM-3, 4호기, 호이스트, 분쇄기2 — startup-load pattern
- Cluster 3: 우측분전반1, L-1 전등 — low-load / environment-dependent pattern

In [ ]:
# Compare April against Dec-Mar stable period
df['month'] = df['localtime'].dt.month
stable = df[df['month'].isin([12,1,2,3])]
april = df[df['month']==4]

rows=[]
for mod in df['module(equipment)'].dropna().unique():
    a=stable[stable['module(equipment)']==mod]
    b=april[april['module(equipment)']==mod]
    if len(a) and len(b):
        rows.append({
            'module':mod,
            'stable_active_mean':a['activePower'].mean(),
            'april_active_mean':b['activePower'].mean(),
            'active_change_pct':(b['activePower'].mean()-a['activePower'].mean())/a['activePower'].mean()*100
        })
pd.DataFrame(rows).sort_values('active_change_pct', ascending=False)